# Submission 

In [1]:
# ============================================================
# ONE CELL — SUBMISSION (FIX ARCH MISMATCH) — final_gate_model.pt (blocks.* compatible)
# - Fix: state_dict "blocks.*" vs "encoder.layers.*"
# - Fix: torch.load PyTorch 2.6 weights_only
# - Fix: pred_features_test.csv not required (fallback search)
# Output: /kaggle/working/submission.csv
# ============================================================

import os, json, math, time, gc, inspect
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import cv2

# ----------------------------
# 0) Paths (USER)
# ----------------------------
FINAL_MODEL_PT = Path("/kaggle/input/recod-ailuc-dinov2-train/recodai_luc_gate_artifacts/model_bundle_v3/final_gate_model.pt")
if not FINAL_MODEL_PT.exists():
    raise FileNotFoundError(f"Model not found: {FINAL_MODEL_PT}")

BUNDLE_DIR = FINAL_MODEL_PT.parent
ART_DIR = BUNDLE_DIR.parent  # .../recodai_luc_gate_artifacts/

print("Using:")
print("  FINAL_MODEL_PT:", FINAL_MODEL_PT)
print("  BUNDLE_DIR    :", BUNDLE_DIR)
print("  ART_DIR       :", ART_DIR)

# ----------------------------
# 1) Competition paths (auto-detect)
# ----------------------------
def find_comp_root():
    pref = Path("/kaggle/input/recodai-luc-scientific-image-forgery-detection")
    if pref.exists():
        return pref
    base = Path("/kaggle/input")
    cands = []
    for d in base.iterdir():
        if d.is_dir() and (d/"sample_submission.csv").exists() and (d/"test_images").exists():
            cands.append(d)
    if not cands:
        raise FileNotFoundError("Competition folder not found under /kaggle/input")
    cands.sort(key=lambda x: (("recod" not in x.name.lower()), x.name))
    return cands[0]

COMP_ROOT = find_comp_root()
SAMPLE_SUB = COMP_ROOT / "sample_submission.csv"
TEST_IMG_DIR = COMP_ROOT / "test_images"
if not SAMPLE_SUB.exists():
    raise FileNotFoundError(f"Missing sample_submission.csv: {SAMPLE_SUB}")

print("  COMP_ROOT   :", COMP_ROOT)
print("  SAMPLE_SUB  :", SAMPLE_SUB)
print("  TEST_IMG_DIR:", TEST_IMG_DIR)

# ----------------------------
# 2) Robust torch.load (PyTorch 2.6 weights_only default True)
# ----------------------------
def torch_load_robust(path: str | Path, map_location="cpu"):
    path = str(path)
    sig = None
    try:
        sig = inspect.signature(torch.load)
    except Exception:
        sig = None

    if sig is not None and "weights_only" in sig.parameters:
        try:
            return torch.load(path, map_location=map_location, weights_only=True)
        except Exception:
            # OK for your own artifact
            return torch.load(path, map_location=map_location, weights_only=False)
    return torch.load(path, map_location=map_location)

# ----------------------------
# 3) Find nearby artifacts
# ----------------------------
def find_nearby(filename: str, roots: list[Path]):
    for r in roots:
        p = r / filename
        if p.exists():
            return p
    for r in roots:
        if r.exists():
            hits = list(r.glob(f"**/{filename}"))
            if hits:
                hits.sort(key=lambda x: len(str(x)))
                return hits[0]
    return None

FEATURE_COLS_JSON = find_nearby("feature_cols.json", [BUNDLE_DIR, ART_DIR, Path("/kaggle/working/recodai_luc_gate_artifacts")])
if FEATURE_COLS_JSON is None:
    raise FileNotFoundError("feature_cols.json not found near bundle. Ensure dataset includes it.")
FEATURE_COLS = json.loads(FEATURE_COLS_JSON.read_text())

THR_JSON = find_nearby("thresholds.json", [BUNDLE_DIR, ART_DIR, Path("/kaggle/working/recodai_luc_gate_artifacts")])
T_GATE = 0.5
if THR_JSON is not None and THR_JSON.exists():
    try:
        thr = json.loads(THR_JSON.read_text())
        if isinstance(thr, dict) and "T_gate" in thr:
            T_GATE = float(thr["T_gate"])
    except Exception:
        pass

print("Artifacts:")
print("  feature_cols:", FEATURE_COLS_JSON)
print("  thresholds  :", THR_JSON if THR_JSON else "(not found -> default 0.5)")
print("  T_GATE      :", T_GATE)
print("  n_features  :", len(FEATURE_COLS))

# ----------------------------
# 4) Load final model packs
# ----------------------------
blob = torch_load_robust(FINAL_MODEL_PT, map_location="cpu")
if not isinstance(blob, dict) or "packs" not in blob:
    raise ValueError(f"Unexpected final_gate_model.pt format. type={type(blob)} keys={list(blob.keys()) if isinstance(blob, dict) else None}")

packs = blob["packs"]
if not isinstance(packs, list) or len(packs) == 0:
    raise ValueError("final_gate_model.pt has empty `packs`.")

cfg = packs[0].get("cfg", None)
if not isinstance(cfg, dict) or len(cfg) == 0:
    raise ValueError("Cannot find cfg inside packs[0]['cfg'].")

print("Gate model packs:", len(packs))
print("cfg keys:", sorted(list(cfg.keys())))

# ----------------------------
# 5) Keys normalization helpers
# ----------------------------
def normalize_state_dict(sd: dict):
    if not isinstance(sd, dict):
        return sd
    # strip common prefixes (module., model.) if present
    for pref in ["module.", "model."]:
        if all(str(k).startswith(pref) for k in sd.keys()):
            sd = {str(k)[len(pref):]: v for k, v in sd.items()}
    # sometimes only some keys have module.
    if any(str(k).startswith("module.") for k in sd.keys()):
        sd = { (str(k)[7:] if str(k).startswith("module.") else str(k)) : v for k, v in sd.items() }
    return sd

def detect_arch(sd: dict):
    keys = list(sd.keys())
    if any(k.startswith("blocks.") for k in keys):
        return "blocks"
    if any(k.startswith("encoder.layers.") for k in keys):
        return "encoder"
    # fallback: infer from presence of final_norm.scale
    if any(k.startswith("final_norm.") for k in keys):
        return "blocks"
    return "encoder"

# ----------------------------
# 6) Build base test table: uid/case_id/variant
# ----------------------------
def ensure_uid_case_variant(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "uid" not in df.columns:
        for alt in ["sample_id", "id", "key"]:
            if alt in df.columns:
                df = df.rename(columns={alt: "uid"})
                break
    if "uid" not in df.columns:
        raise ValueError("Cannot find uid/sample_id column.")
    df["uid"] = df["uid"].astype(str)

    if "case_id" not in df.columns:
        df["case_id"] = df["uid"].str.extract(r"^(\d+)")[0]
    df["case_id"] = df["case_id"].astype(str)

    if "variant" not in df.columns:
        v = df["uid"].str.extract(r"__(\w+)$")[0]
        v2 = df["uid"].str.extract(r"_(\w+)$")[0]
        df["variant"] = v.fillna(v2).fillna("unk").astype(str)

    return df

def find_df_test_parquet():
    roots = [
        Path("/kaggle/working/recodai_luc/artifacts"),
        Path("/kaggle/working"),
        Path("/kaggle/input/recod-ailuc-dinov2-train"),
    ]
    base = Path("/kaggle/input")
    if base.exists():
        for d in base.iterdir():
            if d.is_dir() and any(t in d.name.lower() for t in ["recod", "luc", "dinov2"]):
                roots.append(d)
    cands = []
    for r in roots:
        if r.exists():
            cands.extend(list(r.glob("**/df_test.parquet")))
    if not cands:
        return None
    cands.sort(key=lambda p: (("recodai_luc" not in str(p).lower()), len(str(p))))
    return cands[0]

df_sub = pd.read_csv(SAMPLE_SUB)
df_sub["case_id"] = df_sub["case_id"].astype(str)

DF_TEST_PARQ = find_df_test_parquet()
print("df_test.parquet:", DF_TEST_PARQ if DF_TEST_PARQ else "(not found)")

if DF_TEST_PARQ is not None and DF_TEST_PARQ.exists():
    df_base = pd.read_parquet(DF_TEST_PARQ)
    if "uid" not in df_base.columns:
        if {"case_id","variant"}.issubset(df_base.columns):
            df_base["uid"] = df_base["case_id"].astype(str) + "__" + df_base["variant"].astype(str)
        elif "case_id" in df_base.columns:
            df_base["uid"] = df_base["case_id"].astype(str)
        else:
            df_base = pd.DataFrame({"case_id": df_sub["case_id"].values})
            df_base["uid"] = df_base["case_id"].astype(str)
    keep = ["uid"] + [c for c in ["case_id","variant"] if c in df_base.columns]
    df_base = ensure_uid_case_variant(df_base[keep])
else:
    df_base = pd.DataFrame({"case_id": df_sub["case_id"].values})
    df_base["uid"] = df_base["case_id"].astype(str)
    df_base["variant"] = "unk"
    df_base = ensure_uid_case_variant(df_base)

print("Base test table:", df_base.shape, "| unique cases:", df_base["case_id"].nunique())

# ----------------------------
# 7) Find optional test feature tables (pred/match)
# ----------------------------
def iter_feature_search_roots():
    yield Path("/kaggle/working/recodai_luc/cache")
    yield Path("/kaggle/working")
    yield Path("/kaggle/input/recod-ailuc-dinov2-train")
    base = Path("/kaggle/input")
    if base.exists():
        for d in base.iterdir():
            if d.is_dir() and any(t in d.name.lower() for t in ["recod", "luc", "dinov2"]):
                yield d

def find_feature_file(patterns):
    cands = []
    for root in iter_feature_search_roots():
        if not root.exists():
            continue
        for pat in patterns:
            for p in root.rglob(pat):
                if p.is_file():
                    cands.append(p)
        if len(cands) >= 80:
            break
    if not cands:
        return None
    cands.sort(key=lambda p: (("recodai_luc" not in str(p).lower()), ("cache" not in str(p).lower()), len(str(p))))
    return cands[0]

def load_feat(path: Path) -> pd.DataFrame:
    if path is None:
        return None
    path = Path(path)
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path)

PRED_FEAT_ANY = find_feature_file(["pred_features_test*.csv", "pred_features_test*.parquet"])
MATCH_FEAT_ANY = find_feature_file(["match_features_test*.csv", "match_features_test*.parquet"])

print("Feature candidates:")
print("  pred_features_test* :", PRED_FEAT_ANY if PRED_FEAT_ANY else "(not found)")
print("  match_features_test*:", MATCH_FEAT_ANY if MATCH_FEAT_ANY else "(not found)")

df_feat = df_base.copy()

if PRED_FEAT_ANY is not None:
    try:
        dfp = ensure_uid_case_variant(load_feat(PRED_FEAT_ANY))
        keep = ["uid"] + [c for c in dfp.columns if c not in ["uid","case_id","variant"]]
        df_feat = df_feat.merge(dfp[keep], on="uid", how="left")
        print("Merged pred features:", PRED_FEAT_ANY)
    except Exception as e:
        print("WARN: failed load pred features:", e)

if MATCH_FEAT_ANY is not None:
    try:
        dfm = ensure_uid_case_variant(load_feat(MATCH_FEAT_ANY))
        new_cols = [c for c in dfm.columns if c not in df_feat.columns and c not in ["case_id","variant"]]
        if len(new_cols) > 0:
            df_feat = df_feat.merge(dfm[["uid"] + new_cols], on="uid", how="left")
        print("Merged match features:", MATCH_FEAT_ANY)
    except Exception as e:
        print("WARN: failed load match features:", e)

for c in FEATURE_COLS:
    if c not in df_feat.columns:
        df_feat[c] = 0.0

for c in FEATURE_COLS:
    df_feat[c] = pd.to_numeric(df_feat[c], errors="coerce")

X_test = df_feat[FEATURE_COLS].to_numpy(dtype=np.float32, copy=True)
if not np.isfinite(X_test).all():
    X_test = np.nan_to_num(X_test, nan=0.0, posinf=0.0, neginf=0.0)

nonzero_rate = float(np.mean(np.abs(X_test).sum(axis=1) > 0)) * 100.0
print(f"Feature coverage sanity: rows_with_any_nonzero_feature = {nonzero_rate:.2f}%")
if nonzero_rate < 1.0:
    print("WARN: Hampir semua fitur nol. pred/match_features_test belum ada. Submission tetap jadi, tapi gate tidak informatif.")

# ----------------------------
# 8) Model defs: two variants (encoder vs blocks)
# ----------------------------
class XDataset(Dataset):
    def __init__(self, X):
        self.X = torch.from_numpy(X.astype(np.float32))
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, i): return self.X[i]

def apply_standardizer(X_in: np.ndarray, mu, sig):
    mu = np.asarray(mu, dtype=np.float32)
    sig = np.asarray(sig, dtype=np.float32)
    sig = np.where(sig < 1e-8, 1.0, sig).astype(np.float32)
    return ((X_in - mu) / sig).astype(np.float32)

class FTTransformerEncoder(nn.Module):
    # your original encoder.layers.* version
    def __init__(self, n_features, d_model, n_heads, n_layers, ffn_mult, dropout, attn_dropout):
        super().__init__()
        self.w = nn.Parameter(torch.randn(n_features, d_model) * 0.02)
        self.b = nn.Parameter(torch.zeros(n_features, d_model))
        self.feat_emb = nn.Parameter(torch.randn(n_features, d_model) * 0.02)
        self.cls = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=int(ffn_mult * d_model),
            dropout=float(dropout),
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=int(n_layers))
        self.token_dropout = nn.Dropout(float(attn_dropout))
        self.norm = nn.LayerNorm(d_model)

        self.head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(float(dropout)),
            nn.Linear(d_model, 1),
        )

    def forward(self, x):
        tok = x.unsqueeze(-1) * self.w.unsqueeze(0) + self.b.unsqueeze(0)
        tok = tok + self.feat_emb.unsqueeze(0)
        tok = self.token_dropout(tok)
        B = tok.size(0)
        cls = self.cls.expand(B, -1, -1)
        seq = torch.cat([cls, tok], dim=1)
        z = self.encoder(seq)
        z = self.norm(z[:, 0])
        return self.head(z).squeeze(-1)

class RMSNorm(nn.Module):
    # scale-only norm to match *.scale keys
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(d_model))
        self.eps = eps
    def forward(self, x):
        # x: (B,T,D)
        rms = torch.sqrt(torch.mean(x * x, dim=-1, keepdim=True) + self.eps)
        return (x / rms) * self.scale

class Block(nn.Module):
    def __init__(self, d_model, n_heads, ffn_mult, dropout, attn_dropout):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn = nn.MultiheadAttention(
            embed_dim=int(d_model),
            num_heads=int(n_heads),
            dropout=float(attn_dropout),
            batch_first=True
        )
        self.norm2 = RMSNorm(d_model)
        ffn_dim = int(ffn_mult * d_model)
        # IMPORTANT: indices must match state_dict (ffn.0, ffn.3)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_dim),  # 0
            nn.GELU(),                    # 1
            nn.Dropout(float(dropout)),   # 2
            nn.Linear(ffn_dim, d_model),  # 3
        )
        self.resid_drop = nn.Dropout(float(dropout))

    def forward(self, x):
        # pre-norm attn
        h = self.norm1(x)
        a, _ = self.attn(h, h, h, need_weights=False)
        x = x + self.resid_drop(a)
        # pre-norm ffn
        h = self.norm2(x)
        f = self.ffn(h)
        x = x + self.resid_drop(f)
        return x

class FTTransformerBlocks(nn.Module):
    # blocks.* + final_norm.scale version
    def __init__(self, n_features, d_model, n_heads, n_layers, ffn_mult, dropout, attn_dropout):
        super().__init__()
        self.w = nn.Parameter(torch.randn(n_features, d_model) * 0.02)
        self.b = nn.Parameter(torch.zeros(n_features, d_model))
        self.feat_emb = nn.Parameter(torch.randn(n_features, d_model) * 0.02)
        self.cls = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)

        self.token_dropout = nn.Dropout(float(attn_dropout))
        self.blocks = nn.ModuleList([
            Block(d_model, n_heads, ffn_mult, dropout, attn_dropout) for _ in range(int(n_layers))
        ])
        self.final_norm = RMSNorm(d_model)

        self.head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(float(dropout)),
            nn.Linear(d_model, 1),
        )

    def forward(self, x):
        tok = x.unsqueeze(-1) * self.w.unsqueeze(0) + self.b.unsqueeze(0)
        tok = tok + self.feat_emb.unsqueeze(0)
        tok = self.token_dropout(tok)
        B = tok.size(0)
        cls = self.cls.expand(B, -1, -1)
        z = torch.cat([cls, tok], dim=1)  # (B, 1+F, D)
        for blk in self.blocks:
            z = blk(z)
        z = self.final_norm(z)
        z = z[:, 0]
        return self.head(z).squeeze(-1)

def build_model_for_pack(pack_state_dict: dict, n_features: int, cfg: dict):
    sd = normalize_state_dict(pack_state_dict)
    arch = detect_arch(sd)

    # infer d_model from weight if possible
    d_model = int(cfg.get("d_model", 384))
    if "w" in sd and hasattr(sd["w"], "shape"):
        d_model = int(sd["w"].shape[1])

    n_layers = int(cfg.get("n_layers", 8))
    if arch == "blocks":
        # try infer n_layers from blocks.{i}
        idxs = []
        for k in sd.keys():
            if k.startswith("blocks."):
                try:
                    idxs.append(int(k.split(".")[1]))
                except Exception:
                    pass
        if idxs:
            n_layers = max(idxs) + 1

    model_kwargs = dict(
        n_features=int(n_features),
        d_model=int(d_model),
        n_heads=int(cfg.get("n_heads", 8)),
        n_layers=int(n_layers),
        ffn_mult=int(cfg.get("ffn_mult", 4)),
        dropout=float(cfg.get("dropout", 0.2)),
        attn_dropout=float(cfg.get("attn_dropout", 0.1)),
    )

    if arch == "blocks":
        m = FTTransformerBlocks(**model_kwargs)
    else:
        m = FTTransformerEncoder(**model_kwargs)

    return m, sd, arch

@torch.no_grad()
def predict_proba_one_pack(X_raw: np.ndarray, pack: dict, cfg: dict, batch_size=4096):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = (device.type == "cuda")

    Xn = apply_standardizer(X_raw, pack["mu"], pack["sig"])
    dl = DataLoader(
        XDataset(Xn),
        batch_size=int(batch_size),
        shuffle=False,
        num_workers=2,
        pin_memory=(device.type == "cuda"),
        drop_last=False
    )

    m, sd, arch = build_model_for_pack(pack["state_dict"], X_raw.shape[1], cfg)
    m = m.to(device)

    # first try strict=True (should work if architecture matches)
    try:
        m.load_state_dict(sd, strict=True)
    except Exception as e:
        # fallback strict=False (still run)
        missing, unexpected = m.load_state_dict(sd, strict=False)
        print(f"WARN load_state_dict strict=True failed -> strict=False used | arch={arch}")
        print(f"  missing={len(missing)} | unexpected={len(unexpected)}")
        # show a small hint (not too long)
        if len(missing) > 0:
            print("  missing head:", missing[:5])
        if len(unexpected) > 0:
            print("  unexpected head:", unexpected[:5])

    m.eval()

    ps = []
    for xb in dl:
        xb = xb.to(device, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = m(xb)
            p = torch.sigmoid(logits)
        ps.append(p.detach().cpu().numpy())
    return np.concatenate(ps, axis=0).astype(np.float32)

# ----------------------------
# 9) Gate inference (avg packs)
# ----------------------------
probs_list = []
t0 = time.time()
for i, pk in enumerate(packs):
    for req in ["state_dict", "mu", "sig"]:
        if req not in pk:
            raise ValueError(f"Pack {i} missing key: {req}")
    p = predict_proba_one_pack(X_test, pk, cfg, batch_size=4096)
    probs_list.append(p)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

gate_prob = np.mean(probs_list, axis=0).astype(np.float32)
print(f"OK — Gate inference done | rows={len(gate_prob)} | time={(time.time()-t0):.1f}s")

df_gate = pd.DataFrame({
    "uid": df_feat["uid"].astype(str).values,
    "case_id": df_feat["case_id"].astype(str).values,
    "gate_prob": gate_prob
})

# Case aggregation (default mean)
AGG_MODE = "mean"  # mean / max / p80
if AGG_MODE == "max":
    df_case = df_gate.groupby("case_id", as_index=False)["gate_prob"].max()
elif AGG_MODE == "p80":
    df_case = df_gate.groupby("case_id", as_index=False)["gate_prob"].quantile(0.80)
else:
    df_case = df_gate.groupby("case_id", as_index=False)["gate_prob"].mean()

df_case = df_case.rename(columns={"gate_prob": "case_prob"})
df_case["is_forged"] = (df_case["case_prob"].values >= float(T_GATE)).astype(np.int32)

print("Gate summary:")
print("  AGG_MODE     :", AGG_MODE)
print("  T_GATE       :", T_GATE)
print("  forged_pred% :", float(df_case["is_forged"].mean()) * 100.0)

# ----------------------------
# 10) Optional pred_ens masks (if exists)
# ----------------------------
def find_pred_ens_dir():
    pref = Path("/kaggle/working/recodai_luc/cache/pred_ens")
    if pref.exists():
        return pref
    roots = [Path("/kaggle/input/recod-ailuc-dinov2-train"), Path("/kaggle/input"), Path("/kaggle/working")]
    cands = []
    for r in roots:
        if r.exists():
            cands.extend(list(r.glob("**/pred_ens")))
    if not cands:
        return None
    cands.sort(key=lambda p: (("recodai_luc" not in str(p).lower()), ("cache" not in str(p).lower()), len(str(p))))
    return cands[0]

PRED_ENS_DIR = find_pred_ens_dir()
print("pred_ens dir:", PRED_ENS_DIR if PRED_ENS_DIR else "(not found -> forged will fallback to authentic)")

def rle_encode(mask_u8: np.ndarray):
    pixels = mask_u8.T.flatten()
    dots = np.where(pixels == 1)[0]
    if len(dots) == 0:
        return "authentic"
    run_lengths = []
    prev = -2
    for b in dots:
        if b > prev + 1:
            run_lengths.extend((b + 1, 0))
        run_lengths[-1] += 1
        prev = b
    return json.dumps([int(x) for x in run_lengths])

def _try_unpack_mask_pack(pack: np.ndarray, hw):
    h, w = int(hw[0]), int(hw[1])
    pack = np.asarray(pack)
    if pack.dtype != np.uint8:
        pack = pack.astype(np.uint8, copy=False)
    for bitorder in ["little", "big"]:
        bits = np.unpackbits(pack, bitorder=bitorder)[: h*w]
        if bits.size == h*w:
            m = bits.reshape(h, w).astype(np.uint8)
            s = int(m.sum())
            if 0 < s < h*w:
                return m
    bits = np.unpackbits(pack)[: h*w]
    if bits.size == h*w:
        return bits.reshape(h, w).astype(np.uint8)
    return None

def get_target_hw_from_test(case_id: str):
    p = TEST_IMG_DIR / f"{case_id}.png"
    if not p.exists():
        hits = list(TEST_IMG_DIR.glob(f"{case_id}.*"))
        if hits:
            p = hits[0]
    if not p.exists():
        return None
    try:
        from PIL import Image
        im = Image.open(p)
        return (im.size[1], im.size[0])  # (H,W)
    except Exception:
        return None

def load_mask_from_npz(npz_path: Path, target_hw=None):
    try:
        data = np.load(npz_path, allow_pickle=True)
    except Exception:
        return None, None

    keys = set(data.files)

    for k in ["annotation", "rle", "rle_str", "pred_rle"]:
        if k in keys:
            v = data[k]
            if isinstance(v, np.ndarray) and v.shape == ():
                v = v.item()
            if isinstance(v, bytes):
                v = v.decode("utf-8", errors="ignore")
            if isinstance(v, str) and len(v) > 0:
                return None, v

    for k in ["mask", "mask_u8", "mask_bin", "pred_mask"]:
        if k in keys:
            m = data[k]
            if isinstance(m, np.ndarray):
                if m.ndim == 3:
                    m = m[..., 0]
                m = (m > 0.5).astype(np.uint8)
                if target_hw is not None and (m.shape[0] != target_hw[0] or m.shape[1] != target_hw[1]):
                    m = cv2.resize(m, (int(target_hw[1]), int(target_hw[0])), interpolation=cv2.INTER_NEAREST)
                return m, None

    if "mask_pack" in keys:
        pack = data["mask_pack"]
        hw = None
        for hk in ["hw", "shape", "mask_hw", "orig_hw", "resized_hw", "resized_hw_base", "HW"]:
            if hk in keys:
                v = data[hk]
                if isinstance(v, np.ndarray) and v.shape == ():
                    v = v.item()
                try:
                    vv = np.array(v).reshape(-1).tolist()
                    if len(vv) >= 2:
                        hw = (int(vv[0]), int(vv[1]))
                        break
                except Exception:
                    pass
        if hw is None and target_hw is not None:
            hw = target_hw
        if hw is not None:
            m = _try_unpack_mask_pack(pack, hw)
            if m is not None:
                if target_hw is not None and (m.shape[0] != target_hw[0] or m.shape[1] != target_hw[1]):
                    m = cv2.resize(m, (int(target_hw[1]), int(target_hw[0])), interpolation=cv2.INTER_NEAREST)
                return m, None

    return None, None

def get_case_annotation(case_id: str, is_forged: int):
    if is_forged == 0:
        return "authentic"
    if PRED_ENS_DIR is None:
        return "authentic"

    cand = PRED_ENS_DIR / f"{case_id}.npz"
    if not cand.exists():
        hits = list(PRED_ENS_DIR.glob(f"{case_id}*.npz"))
        if hits:
            hits.sort(key=lambda x: x.name)
            cand = hits[0]
        else:
            return "authentic"

    target_hw = get_target_hw_from_test(case_id)
    m, rle_direct = load_mask_from_npz(cand, target_hw=target_hw)
    if isinstance(rle_direct, str) and len(rle_direct) > 0:
        return rle_direct
    if m is None:
        return "authentic"
    return rle_encode(m)

# ----------------------------
# 11) Build submission
# ----------------------------
sub = pd.read_csv(SAMPLE_SUB)
if not {"case_id", "annotation"}.issubset(sub.columns):
    raise ValueError(f"sample_submission columns unexpected: {list(sub.columns)}")
sub["case_id"] = sub["case_id"].astype(str)

sub = sub.merge(df_case[["case_id","case_prob","is_forged"]], on="case_id", how="left")
sub["is_forged"] = sub["is_forged"].fillna(0).astype(int)

anns = [get_case_annotation(cid, int(fg)) for cid, fg in zip(sub["case_id"].values, sub["is_forged"].values)]
sub["annotation"] = anns
sub = sub[["case_id","annotation"]]

out_path = Path("/kaggle/working/submission.csv")
sub.to_csv(out_path, index=False)

print("\nOK — submission saved:", out_path)
print("  rows       :", len(sub))
print("  authentic% :", float((sub["annotation"]=="authentic").mean())*100.0)
display(sub.head(10))


Using:
  FINAL_MODEL_PT: /kaggle/input/recod-ailuc-dinov2-train/recodai_luc_gate_artifacts/model_bundle_v3/final_gate_model.pt
  BUNDLE_DIR    : /kaggle/input/recod-ailuc-dinov2-train/recodai_luc_gate_artifacts/model_bundle_v3
  ART_DIR       : /kaggle/input/recod-ailuc-dinov2-train/recodai_luc_gate_artifacts
  COMP_ROOT   : /kaggle/input/recodai-luc-scientific-image-forgery-detection
  SAMPLE_SUB  : /kaggle/input/recodai-luc-scientific-image-forgery-detection/sample_submission.csv
  TEST_IMG_DIR: /kaggle/input/recodai-luc-scientific-image-forgery-detection/test_images
Artifacts:
  feature_cols: /kaggle/input/recod-ailuc-dinov2-train/recodai_luc_gate_artifacts/model_bundle_v3/feature_cols.json
  thresholds  : /kaggle/input/recod-ailuc-dinov2-train/recodai_luc_gate_artifacts/model_bundle_v3/thresholds.json
  T_GATE      : 0.5
  n_features  : 62
Gate model packs: 1
cfg keys: ['attn_dropout', 'batch_size', 'd_model', 'dropout', 'epochs', 'ffn_mult', 'grad_clip', 'lr', 'min_delta', 'n_head

/tmp/ipykernel_17/1141848632.py:184: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["variant"] = v.fillna(v2).fillna("unk").astype(str)


Feature candidates:
  pred_features_test* : (not found)
  match_features_test*: /kaggle/input/recod-ailuc-dinov2-base/recodai_luc/cache/match_base_cfg_f9f7ea3a65c5/match_features_test.csv
Merged match features: /kaggle/input/recod-ailuc-dinov2-base/recodai_luc/cache/match_base_cfg_f9f7ea3a65c5/match_features_test.csv
Feature coverage sanity: rows_with_any_nonzero_feature = 100.00%


/tmp/ipykernel_17/1141848632.py:184: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["variant"] = v.fillna(v2).fillna("unk").astype(str)
/tmp/ipykernel_17/1141848632.py:524: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


OK — Gate inference done | rows=1 | time=0.6s
Gate summary:
  AGG_MODE     : mean
  T_GATE       : 0.5
  forged_pred% : 100.0
pred_ens dir: (not found -> forged will fallback to authentic)

OK — submission saved: /kaggle/working/submission.csv
  rows       : 1
  authentic% : 100.0


,case_id,annotation
0,45,authentic
